RDS (Relational Database Service) is AWS's managed service for running relational databases without managing the underlying infrastructure. Aurora is AWS's cloud-native relational engine — MySQL and PostgreSQL compatible but built from scratch for higher performance, availability, and scale. Together they cover virtually every relational database need on AWS and are heavily tested on the SAA-C03 exam.

## RDS Overview

RDS manages the undifferentiated heavy lifting of running a relational database: OS patching, DB engine upgrades, backups, storage provisioning, and hardware replacement. You focus on schema design and queries.

### Supported engines
- MySQL
- PostgreSQL
- MariaDB
- Oracle
- Microsoft SQL Server
- IBM Db2

### RDS vs self-managed on EC2

| | RDS | EC2 + DB engine |
|---|---|---|
| OS access | No SSH | Full access |
| Patching | AWS handles | You handle |
| Backups | Automated | You configure |
| Multi-AZ failover | One checkbox | Complex setup |
| Read replicas | Built-in | You build |
| Cost | Higher | Lower (but OpEx) |

> Use EC2 only if you need OS-level access for a feature RDS does not support (e.g. custom plugins, Oracle RAC).

## RDS Storage

RDS uses EBS volumes for storage. Three volume types:

| Type | Use case |
|---|---|
| **gp2** | Legacy general purpose; IOPS scale with size |
| **gp3** | Recommended default; independent IOPS/throughput provisioning |
| **io1** | High IOPS workloads (> 64,000 IOPS needed) |

### Storage autoscaling
Enable **RDS Storage Autoscaling** to automatically increase storage when free space falls below a threshold — no manual intervention, no downtime. You set a maximum storage limit. Useful for unpredictable growth workloads.

## Backups and Snapshots

### Automated backups
- Enabled by default
- Daily full backup during the maintenance window + transaction log backup every 5 minutes
- Retention: **1 to 35 days** (0 = disabled)
- Enables **point-in-time restore (PITR)** — restore to any second within the retention window
- Backups stored in S3 (managed by AWS, not visible in your S3 console)
- Deleted when the RDS instance is deleted (unless you take a final snapshot)

### Manual snapshots
- Triggered by you (or automation)
- **Persist indefinitely** — not deleted when the DB instance is deleted
- Can be copied to another region for DR
- Can be shared with other AWS accounts
- Used to restore as a new DB instance

### Restore behaviour
Restoring from a backup or snapshot always creates a **new RDS instance** — it does not overwrite the existing one. You then update your application's connection string to point to the new instance.

## Multi-AZ Deployments

Multi-AZ creates a **synchronous standby replica** in a different AZ. It is purely for **high availability** — not for read scaling.

```text
App → RDS Primary (AZ-a)
              │  synchronous replication
              ▼
       Standby (AZ-b)  ← no traffic, just waiting
```

### How failover works
1. AWS detects primary failure (hardware, AZ outage, OS crash, maintenance)
2. DNS record for the DB endpoint is updated to point to the standby
3. Standby is promoted to primary
4. Typically completes in **1–2 minutes**
5. Your application reconnects using the same endpoint DNS name

### Key points
- Standby receives **no read traffic** — it is hot standby only
- Backups are taken from the standby to avoid I/O impact on primary
- Multi-AZ is a single-region feature — for cross-region HA, use Aurora Global or read replica promotion
- **Multi-AZ DB Cluster** (newer): 1 writer + 2 readers across 3 AZs — provides both HA and limited read scaling

> **Exam tip:** Multi-AZ = availability. Read Replicas = scalability. Do not confuse them.

## Read Replicas

Read Replicas use **asynchronous replication** to offload read traffic from the primary instance.

```text
Write traffic → RDS Primary
                    │  async replication
                    ├──▶ Read Replica 1  ← read traffic
                    ├──▶ Read Replica 2  ← read traffic
                    └──▶ Read Replica 3  (different region)
```

### Key properties
- Up to **5 read replicas** per RDS instance (15 for Aurora)
- Can be in the **same AZ, different AZ, or different region**
- **Asynchronous** — small replication lag; reads may return slightly stale data
- Read replicas have their own DNS endpoint — your app must explicitly direct reads to them
- Can be **promoted** to a standalone DB (breaks replication — used for DR or migrations)
- Cross-region read replicas: replication traffic costs money; useful for disaster recovery

### Replication cost
- **Same region**: free
- **Cross-region**: standard data transfer charges apply

### Multi-AZ vs Read Replicas

| | Multi-AZ | Read Replica |
|---|---|---|
| Purpose | High availability | Read scaling / DR |
| Replication | Synchronous | Asynchronous |
| Serves reads? | No | Yes |
| Automatic failover | Yes | No (manual promotion) |
| Endpoint | Same endpoint | Separate endpoint |
| Cross-region | No | Yes |

## RDS Security

### Network
- RDS instances are deployed **inside a VPC** in private subnets (recommended)
- Security groups control inbound access (e.g. allow port 5432 from app-server SG only)
- No direct public internet access needed — application tier accesses DB over private network

### Encryption
- **At rest**: AES-256 via KMS. Must be enabled at creation — cannot be added later.
  - To encrypt an unencrypted instance: snapshot → copy snapshot with encryption → restore
- **In transit**: SSL/TLS enforced via parameter group settings or connection string flags
- Read replicas of an encrypted primary are automatically encrypted with the same key

### IAM authentication
Instead of username/password, use **IAM DB authentication**:
- Supported on MySQL and PostgreSQL
- Application requests a short-lived auth token from IAM (valid 15 minutes)
- Token used instead of password — no password stored in app config
- Works well with EC2 instance profiles and Lambda execution roles

### Secrets Manager integration
Store DB credentials in Secrets Manager with **automatic rotation**. RDS can rotate the secret on a schedule — your application fetches the current secret from Secrets Manager rather than hard-coding credentials.

## RDS Proxy

**RDS Proxy** sits between your application and RDS, maintaining a **pool of connections** to the database.

```text
Lambda functions (thousands of concurrent)
        │  many short-lived connections
        ▼
    RDS Proxy  ←── connection pool (few persistent connections)
        │
        ▼
    RDS / Aurora
```

### Why it matters
Relational databases have a hard limit on concurrent connections. Lambda functions are ephemeral — each invocation opens a new DB connection. At scale (thousands of concurrent Lambdas) this overwhelms the DB.

RDS Proxy multiplexes thousands of application connections into a small pool of long-lived DB connections.

### Key features
- Reduces DB connection overhead by up to **66%**
- **Automatic failover** — RDS Proxy detects Multi-AZ failover and reconnects to the new primary in seconds; application connections are preserved
- **IAM authentication** and **Secrets Manager** integration — credentials never leave Proxy
- Fully **serverless** and auto-scaling
- **VPC-only** — Proxy endpoint is not publicly accessible

> **Exam tip:** Lambda + RDS = always put RDS Proxy in between.

## Amazon Aurora

Aurora is AWS's cloud-native relational database engine — purpose-built for the cloud, not a port of an existing engine. It is **MySQL and PostgreSQL compatible** but reimplements the storage and replication layer.

### Performance
- **5× faster** than standard MySQL on RDS
- **3× faster** than standard PostgreSQL on RDS
- Achieved through a distributed, log-structured storage system that eliminates many traditional DB bottlenecks

### Aurora storage architecture
Aurora separates compute from storage:

```text
Aurora DB instances (compute)
        │
        ▼
Aurora Cluster Volume (shared storage)
  ├── AZ-1: copy 1, copy 2
  ├── AZ-2: copy 3, copy 4
  └── AZ-3: copy 5, copy 6
```

- **6 copies of data** across 3 AZs automatically
- Writes require **4 out of 6** copies to acknowledge
- Reads require **3 out of 6** copies
- **Self-healing**: Aurora continuously scans for corrupted blocks and repairs them using other copies
- Storage **auto-grows** in 10 GB increments up to **128 TB** — no manual provisioning
- You are billed per GB actually used, not pre-provisioned

### Failover speed
Because Aurora compute and storage are separated, failover promotes a read replica to primary without any storage rebuild — typically **< 30 seconds** vs 1–2 minutes for RDS Multi-AZ.

## Aurora Endpoints

An Aurora cluster exposes multiple endpoint types:

| Endpoint | Routes to | Use case |
|---|---|---|
| **Cluster endpoint** | Current primary instance | All write traffic |
| **Reader endpoint** | Load balances across all read replicas | Read traffic |
| **Custom endpoint** | A subset of instances you define | Different replica sizes for different workloads |
| **Instance endpoint** | A specific instance | Debugging, maintenance |

```text
Writes  → cluster endpoint  → Primary
Reads   → reader endpoint   → Replica 1, Replica 2, Replica 3 (load balanced)
Reports → custom endpoint   → Replica 4, Replica 5 (large instance class)
```

Up to **15 Aurora read replicas** — vs 5 for RDS. All replicas share the same storage volume — replication is done at the storage layer, not by sending binary logs.

## Aurora Advanced Features

### Aurora Serverless v2
Aurora Serverless v2 automatically scales Aurora's compute capacity up and down in fine-grained increments (0.5 ACU steps) based on actual load — within seconds.

- **ACU** (Aurora Capacity Unit) = ~2 GB RAM + CPU
- Set a minimum and maximum ACU range
- Scales smoothly — no connection drops during scaling
- Ideal for: variable workloads, dev/test, multi-tenant SaaS, new applications with unknown scale
- Can mix Serverless v2 instances and provisioned instances in the same cluster

### Aurora Global Database
Spans **multiple AWS regions** with a single Aurora cluster:

```text
Primary region (us-east-1)  → Writes
        │  replication lag < 1 second
        ▼
Secondary region (eu-west-1) → Reads
Secondary region (ap-east-1) → Reads
```

- Replication lag **< 1 second** (storage-level replication, not binary logs)
- **RPO < 1 second**, **RTO < 1 minute** for cross-region DR
- Up to 5 secondary regions, each with up to 16 read replicas
- **Managed failover**: promote a secondary region to primary in under a minute
- Use case: globally distributed applications, DR across regions, local read latency

### Aurora Backtrack
**Rewind the database** to a previous point in time **without restoring from a snapshot**:
- Available for Aurora MySQL only
- Backtrack window: up to 72 hours
- Takes seconds — much faster than a full restore
- Use case: accidentally deleted data, bad migration, testing — quickly undo without spinning up a new instance

### Aurora Multi-Master
All instances can accept **write traffic** — no single primary. If one writer fails, others continue immediately. Available for Aurora MySQL. Use case: applications requiring continuous write availability.

## RDS vs Aurora Decision Guide

| Scenario | Choose |
|---|---|
| Need Oracle or SQL Server | RDS (Aurora doesn't support these) |
| MySQL/PostgreSQL, need maximum performance | Aurora |
| MySQL/PostgreSQL, cost-sensitive, predictable workload | RDS |
| Variable/unpredictable workload, scale to zero | Aurora Serverless v2 |
| Cross-region DR with < 1s RPO | Aurora Global Database |
| Serverless + relational DB | Aurora Serverless v2 |
| Lambda connecting to relational DB | RDS Proxy (any engine) |
| Need to undo accidental data change fast | Aurora Backtrack (MySQL) |
| Multi-region active-active writes | Aurora Multi-Master |

## Working with RDS and Aurora via boto3

In [ ]:
import boto3

rds = boto3.client('rds', region_name='us-east-1')

In [ ]:
# Create a Multi-AZ RDS PostgreSQL instance
rds.create_db_instance(
    DBInstanceIdentifier='prod-postgres',
    DBInstanceClass='db.t3.medium',
    Engine='postgres',
    EngineVersion='16.1',
    MasterUsername='admin',
    MasterUserPassword='change-me-use-secrets-manager',
    AllocatedStorage=100,          # GB
    StorageType='gp3',
    StorageEncrypted=True,
    MultiAZ=True,                  # synchronous standby in another AZ
    DBSubnetGroupName='my-db-subnet-group',
    VpcSecurityGroupIds=['sg-0dbaccess'],
    BackupRetentionPeriod=7,       # 7 days PITR
    EnableIAMDatabaseAuthentication=True,
    DeletionProtection=True,
    Tags=[{'Key': 'Environment', 'Value': 'production'}]
)
print("RDS PostgreSQL Multi-AZ instance creation initiated")

In [ ]:
# Create a read replica in the same region
rds.create_db_instance_read_replica(
    DBInstanceIdentifier='prod-postgres-replica-1',
    SourceDBInstanceIdentifier='prod-postgres',
    DBInstanceClass='db.t3.medium',
    AvailabilityZone='us-east-1b',
    StorageEncrypted=True
)
print("Read replica creation initiated")

In [ ]:
# Create an Aurora PostgreSQL cluster with Serverless v2
rds.create_db_cluster(
    DBClusterIdentifier='prod-aurora-cluster',
    Engine='aurora-postgresql',
    EngineVersion='16.1',
    MasterUsername='admin',
    MasterUserPassword='change-me-use-secrets-manager',
    DatabaseName='appdb',
    DBSubnetGroupName='my-db-subnet-group',
    VpcSecurityGroupIds=['sg-0dbaccess'],
    StorageEncrypted=True,
    BackupRetentionPeriod=7,
    EnableIAMDatabaseAuthentication=True,
    ServerlessV2ScalingConfiguration={
        'MinCapacity': 0.5,   # ACUs — scales down to near-zero
        'MaxCapacity': 16.0   # ACUs — scales up to handle load
    },
    DeletionProtection=True
)

# Add a Serverless v2 writer instance to the cluster
rds.create_db_instance(
    DBInstanceIdentifier='prod-aurora-writer',
    DBClusterIdentifier='prod-aurora-cluster',
    DBInstanceClass='db.serverless',   # Serverless v2 instance class
    Engine='aurora-postgresql'
)
print("Aurora Serverless v2 cluster created")

In [ ]:
# Create an RDS Proxy in front of the Aurora cluster
rds.create_db_proxy(
    DBProxyName='aurora-proxy',
    EngineFamily='POSTGRESQL',
    Auth=[
        {
            'AuthScheme': 'SECRETS',
            'SecretArn': 'arn:aws:secretsmanager:us-east-1:123456789012:secret:aurora-creds',
            'IAMAuth': 'REQUIRED'   # require IAM auth — no password needed by app
        }
    ],
    RoleArn='arn:aws:iam::123456789012:role/rds-proxy-role',
    VpcSubnetIds=['subnet-0priv1a', 'subnet-0priv1b'],
    VpcSecurityGroupIds=['sg-0proxy'],
    RequireTLS=True
)
print("RDS Proxy created — Lambda functions should connect via proxy endpoint")

In [ ]:
# Point-in-time restore — creates a NEW instance
rds.restore_db_instance_to_point_in_time(
    SourceDBInstanceIdentifier='prod-postgres',
    TargetDBInstanceIdentifier='prod-postgres-restored',
    RestoreTime='2026-04-15T10:30:00Z',  # any second within retention window
    DBInstanceClass='db.t3.medium',
    MultiAZ=False  # restore as single-AZ for investigation
)
print("PITR restore initiated — update connection string when ready")

## Summary

| Concept | Key Takeaway |
|---|---|
| RDS engines | MySQL, PostgreSQL, MariaDB, Oracle, SQL Server, Db2 |
| Automated backups | 1–35 day retention; enables PITR to any second |
| Manual snapshots | Persist until deleted; survives instance deletion |
| Restore | Always creates a new instance — never overwrites existing |
| Multi-AZ | Synchronous standby; automatic failover 1–2 min; no reads; HA only |
| Read Replicas | Async replication; serves reads; manual promotion; cross-region supported |
| Cross-region replica cost | Data transfer charges apply |
| Encrypt unencrypted RDS | Snapshot → copy with encryption → restore |
| IAM DB auth | Token-based; no password in app; 15-min token TTL |
| RDS Proxy | Connection pooling for Lambda/serverless; faster failover; IAM auth |
| Aurora storage | 6 copies across 3 AZs; self-healing; auto-grows to 128 TB |
| Aurora failover | < 30 seconds (vs 1–2 min for RDS Multi-AZ) |
| Aurora read replicas | Up to 15; storage-layer replication (no lag overhead) |
| Aurora endpoints | Cluster (write), Reader (read LB), Custom (subset), Instance |
| Aurora Serverless v2 | Fine-grained auto-scaling; 0.5 ACU steps; no connection drops |
| Aurora Global DB | < 1s replication lag; RPO < 1s; RTO < 1 min; up to 5 regions |
| Aurora Backtrack | Rewind MySQL DB without restore; up to 72h; seconds to execute |
| Aurora vs RDS | Aurora for MySQL/PG with better perf/HA; RDS for Oracle/SQL Server |